In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# Trigger-Aware BFS agent (exp005, D4 of SPEC_4WEEKS).
# Submitted as an LB ablation data point: random+state-graph-dedup vs
# FORGE+CNN. Pure-Python; no torch.
# Local source-of-truth lives in agents/state_graph.py and
# agents/trigger_bfs_agent.py; this file is the inlined Kaggle copy.
# =====================================================================
from __future__ import annotations

import contextlib
import hashlib
import logging
import random as _random
from collections import deque
from dataclasses import dataclass, field
from typing import Any

import numpy as np

from agents.agent import Agent
from arcengine import GameAction, GameState

logger = logging.getLogger(__name__)

# --------------------------------------------------------------------
# State graph primitives (mirror of agents/state_graph.py)
# --------------------------------------------------------------------


def _hash_frame(frame_layers) -> bytes:
    h = hashlib.blake2b(digest_size=8)
    if frame_layers is None:
        return h.digest()
    for layer in frame_layers:
        tobytes = getattr(layer, "tobytes", None)
        if callable(tobytes):
            h.update(tobytes())
        else:
            for row in layer:
                h.update(bytes(int(v) % 256 for v in row))
    return h.digest()


@dataclass
class StateNode:
    state_hash: bytes
    visit_count: int = 0
    untried_actions: set = field(default_factory=set)
    edges: dict = field(default_factory=dict)
    last_score: int = 0
    last_levels: int = 0
    incoming_change_score: float = 0.0


class StateGraph:
    def __init__(self) -> None:
        self.nodes: dict = {}
        self.frontier: deque = deque()
        self.action_history: list = []
        self.current_levels: int = 0

    def reset(self) -> None:
        self.nodes.clear()
        self.frontier.clear()
        self.action_history.clear()

    def maybe_reset_for_level(self, levels: int) -> bool:
        if levels != self.current_levels:
            self.reset()
            self.current_levels = levels
            return True
        return False

    def add_or_get(self, state_hash, available_actions, levels, score=0):
        node = self.nodes.get(state_hash)
        if node is None:
            untried = {int(a) for a in (available_actions or []) if int(a) != 0}
            node = StateNode(
                state_hash=state_hash, untried_actions=untried,
                last_levels=levels, last_score=score,
            )
            self.nodes[state_hash] = node
            if untried:
                self.frontier.append(state_hash)
        return node

    def observe(self, prev_hash, action_id, next_hash, change_score=0.0):
        if prev_hash is not None and prev_hash in self.nodes:
            node = self.nodes[prev_hash]
            node.edges[action_id] = next_hash
            node.untried_actions.discard(action_id)
            if not node.untried_actions:
                with contextlib.suppress(ValueError):
                    self.frontier.remove(prev_hash)
        if next_hash in self.nodes:
            self.nodes[next_hash].visit_count += 1
            self.nodes[next_hash].incoming_change_score = change_score

    def record_action(self, state_hash, action_id, data):
        self.action_history.append((state_hash, action_id, dict(data)))


# --------------------------------------------------------------------
# Helper: trigger score and click sampling
# --------------------------------------------------------------------


def _to_ndarray(layer):
    if isinstance(layer, np.ndarray):
        return layer
    return np.asarray(layer, dtype=np.uint8)


def _trigger_score(prev_layers, next_layers, prev_levels, next_levels):
    if prev_layers is None or next_layers is None:
        return 0.0
    p = _to_ndarray(prev_layers[-1])
    n = _to_ndarray(next_layers[-1])
    if p.shape != n.shape:
        return float(5 * (next_levels - prev_levels))
    delta_pixels = float((p != n).sum())
    new_colors = float(len({int(v) for v in n.flat} - {int(v) for v in p.flat}))
    delta_levels = float(next_levels - prev_levels)
    return delta_pixels + 5.0 * delta_levels + 2.0 * new_colors


def _sample_click_xy(layers, rng):
    try:
        if not layers:
            raise ValueError("no layers")
        grid = _to_ndarray(layers[-1])
        if grid.ndim != 2:
            raise ValueError("bad grid")
        bg = int(np.bincount(grid.flatten(), minlength=16).argmax())
        ys, xs = np.where(grid != bg)
        if len(xs) > 0:
            idx = rng.randrange(len(xs))
            return {"x": int(xs[idx]), "y": int(ys[idx])}
    except Exception:  # noqa: S110
        pass
    return {"x": rng.randint(0, 63), "y": rng.randint(0, 63)}


# --------------------------------------------------------------------
# MyAgent: the class the harness loads on Kaggle.
# Required surface: subclass of agents.agent.Agent, and dual-arg
# choose_action(frames, latest_frame) + is_done(frames, latest_frame).
# --------------------------------------------------------------------


class MyAgent(Agent):
    """Trigger-Aware BFS agent (state-graph + uniform-random over untried)."""

    MAX_ACTIONS = 80

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._rng = _random.Random(0)
        self.graph = StateGraph()
        self._prev_hash = None
        self._prev_action = None
        self._prev_data: dict = {}
        self._prev_layers = None
        self._prev_levels = 0

    def is_done(self, frames, latest_frame) -> bool:
        return latest_frame.state == GameState.WIN

    def choose_action(self, frames, latest_frame):
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self._prev_hash = None
            self._prev_action = None
            self._prev_layers = None
            return GameAction.RESET

        cur_layers = list(getattr(latest_frame, "frame", []) or [])
        cur_hash = _hash_frame(cur_layers) if cur_layers else b"\x00" * 8
        cur_levels = int(getattr(latest_frame, "levels_completed", 0))

        self.graph.maybe_reset_for_level(cur_levels)

        change_score = _trigger_score(
            self._prev_layers, cur_layers, self._prev_levels, cur_levels
        )

        avail = list(
            getattr(latest_frame, "available_actions", []) or [1, 2, 3, 4, 5, 6, 7]
        )
        node = self.graph.add_or_get(
            cur_hash, available_actions=avail, levels=cur_levels
        )

        if self._prev_hash is not None and self._prev_action is not None:
            self.graph.observe(self._prev_hash, self._prev_action, cur_hash, change_score)

        non_reset_avail = [int(a) for a in avail if int(a) != 0]
        if not non_reset_avail:
            non_reset_avail = [1, 2, 3, 4, 5, 6, 7]

        # Random-uniform over untried; fall back to highest-change-score edge,
        # then to uniform over all.
        untried_avail = [a for a in node.untried_actions if a in non_reset_avail]
        if untried_avail:
            chosen = self._rng.choice(untried_avail)
        else:
            scored = []
            for a, succ_h in node.edges.items():
                if a not in non_reset_avail:
                    continue
                succ = self.graph.nodes.get(succ_h)
                if succ is None:
                    continue
                s = succ.incoming_change_score + 0.05 * len(succ.untried_actions)
                scored.append((s, a))
            if scored and max(s for s, _ in scored) > 0.0:
                top = max(scored)[0]
                top_actions = [a for s, a in scored if s == top]
                chosen = self._rng.choice(top_actions)
            else:
                chosen = self._rng.choice(non_reset_avail)

        action = GameAction.from_id(chosen)
        data: dict = {}
        if action.is_complex():
            data = _sample_click_xy(cur_layers, self._rng)
            action.set_data(data)

        self._prev_hash = cur_hash
        self._prev_action = chosen
        self._prev_data = data
        self._prev_layers = cur_layers
        self._prev_levels = cur_levels
        self.graph.record_action(cur_hash, chosen, data)
        return action


this only runs if you submit to the competition, not when you do tests

In [ ]:
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py','w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
""")
    with open('/kaggle/working/ARC-AGI-3-Agents/.env','w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score'])
    submission.to_parquet('/kaggle/working/submission.parquet',index=False)

This is a dummy submission fallback, important to keep